# CryptoGraph-Secure: Headless Colab Runner

Standardized runner executing model training jobs with deterministic seeds and SHA-256 data validation.

In [ ]:
# Cell 1: Clone repository securely using Colab Secret
import os, subprocess
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_owner = 'da-ksh-esh'
repo_name = 'CryptoGraph'

if not os.path.exists(repo_name):
    clone_url = f'https://{token}@github.com/{repo_owner}/{repo_name}.git'
    subprocess.run(['git', 'clone', clone_url, repo_name], check=True)
    # Scrub credentials from git remote URL immediately
    subprocess.run(['git', '-C', repo_name, 'remote', 'set-url', 'origin', f'https://github.com/{repo_owner}/{repo_name}.git'], check=True)
    print("Cloned repository and scrubbed token from remote configuration.")
else:
    print("Repository already exists locally.")

In [ ]:
# Cell 2: Install dependencies in editable mode
%cd /content/CryptoGraph
!pip install -e . --no-deps

In [ ]:
# Cell 3: Mount Google Drive and unpack verified dataset
from google.colab import drive
drive.mount('/content/drive')

# Unpack verified processed package
zip_source = '/content/drive/MyDrive/CryptoGraph/data/v1.zip'
if os.path.exists(zip_source):
    !python -m cryptograph.pipelines.package_data unpack {zip_source} --target-dir data/processed
else:
    print(f"Package not found at {zip_source}. Please verify drive path or use local mock.")

In [ ]:
# Cell 4: Execute headless training
!python -m cryptograph.pipelines.train --config configs/experiment/tgn_daily_baseline.yaml --seed 42 --device cuda

In [ ]:
# Cell 5: Sync artifacts and results.csv to Google Drive
!cp -r runs /content/drive/MyDrive/CryptoGraph/
print("Experiment runs synced to Google Drive successfully.")